# RI-JK UHF Hessian 分解概览

本文档与 `01-decomp_nh3_r.ipynb` 类似，但针对 unrestricted Hartree-Fock (UHF) 的 Hessian 分解。

与 RHF 的主要差异：

- 分子轨道系数、占据数、能量都按 $\alpha/\beta$ 两套自旋通道存储：`mo_coeff.shape = (2, nao, nmo)`、`mo_occ.shape = (2, nmo)`、`mo_energy.shape = (2, nmo)`。
- 密度矩阵为 $D = D^\alpha + D^\beta$，其中 $D^\sigma = C_\text{occ}^\sigma (C_\text{occ}^\sigma)^T$（不再带 RHF 中的因子 2）。
- 库伦贡献 $E_J$ 形式上与 RHF 一致（依赖总密度 $D$），但 K 部分变成两个自旋通道分别贡献 $K^\alpha[D^\alpha] + K^\beta[D^\beta]$。
- 在 PySCF 的 `pyscf.df.hessian.uhf._partial_hess_ejk` 中，返回的 `ek` 已经包含两个自旋通道之和（且无 RHF 中的额外 $1/2$ 抵消），因此最终公式中 K 前的系数为 $-1$（而非 RHF 的 $-1/2$）。

In [1]:
from pyscf import gto, scf, lib
import numpy as np
from pyscf.hessian import rhf as rhf_hess
from pyscf.hessian import uhf as uhf_hess
from pyscf.df.hessian import uhf as df_uhf_hess
from functools import partial

lib.num_threads(16)
np.set_printoptions(5, suppress=True, linewidth=150)
np.einsum = partial(np.einsum, optimize="greedy")

In [2]:
xyz = """
N  0   0   0
H  1.0 0.1 0.2
H  0.3 1.1 0.2
H  0.1 0.1 1.2
"""

mol = gto.Mole(atom=xyz, basis="def2-TZVP", charge=2, spin=2, max_memory=32000).build()

In [3]:
mf = scf.UHF(mol).density_fit()
mf.mo_coeff = np.load("nh3_u_hf.npz")["mo_coeff"]
mf.mo_occ = np.load("nh3_u_hf.npz")["mo_occ"]
mf.mo_energy = np.load("nh3_u_hf.npz")["mo_energy"]
mf.with_df.build()
mf.converged = True

In [4]:
mf_hess = mf.Hessian().run()
de_ref = mf_hess.de.copy()
print("de_ref shape:", de_ref.shape)
assert np.allclose(de_ref, np.load("nh3_u_hf.npz")["ref_de"])

de_ref shape: (4, 4, 3, 3)


## Hessian 分解概览

与 RHF 的分解结构完全一致，仍然分为以下五个部分（计算量大致由小到大）：

1. 密度矩阵非依赖项：原子核排斥能的导数。该部分与 RHF 完全相同。
2. 密度矩阵一阶项导数（hcore 贡献）。形式上与 RHF 相同，因为依赖的是总密度 $D = D^\alpha + D^\beta$。
3. 重叠积分导数贡献。来自 U 矩阵占据部分化简而来的能量加权密度矩阵；UHF 中能量加权密度矩阵需要分别按 $\alpha/\beta$ 通道累加。
4. 复杂能量二阶 skeleton 导数：UHF 中即 $E_J$（依赖总密度）与 $E_K^\alpha + E_K^\beta$（按自旋分别构造）。
5. CP-HF 贡献。UHF 的 CP-HF 方程在 $\alpha/\beta$ 两通道上分别有 U 矩阵 `mo1a`、`mo1b`，两者相互耦合。

In [5]:
# 1. Density matrix independent term
# 核排斥项不依赖电子结构，与 RHF 完全相同。
de_nuc = rhf_hess.hess_nuc(mol)

In [6]:
# 2. Density matrix first-order term (hcore contribution)
# 3. Overlap integral derivative contribution
# 2 and 3 are computed together to `de_1`.

# auxbasis_response = 0: only orbital derivatives (basis_2nd)
hessobj_aux0 = mf.Hessian()
hessobj_aux0.auxbasis_response = 0
de_1, ej_aux0, ek_aux0 = df_uhf_hess._partial_hess_ejk(hessobj_aux0)

In [7]:
# auxbasis_response = 1: 1st-order aux response (hessian contribution is scaled by 0.5)
hessobj_aux1 = mf.Hessian()
hessobj_aux1.auxbasis_response = 1
_, ej_aux1, ek_aux1 = df_uhf_hess._partial_hess_ejk(hessobj_aux1)

In [8]:
# auxbasis_response = 2: full aux response
hessobj_aux2 = mf.Hessian()
hessobj_aux2.auxbasis_response = 2
_, ej_aux2, ek_aux2 = df_uhf_hess._partial_hess_ejk(hessobj_aux2)

In [9]:
# 4. J/K contribution
# 与 RHF 一样，auxbasis_response==1 时贡献会乘 0.5；通过差分提取 (20)/(11)/(02) 三个阶数。
de_J20 = ej_aux0.copy()
de_J11 = 2.0 * (ej_aux1 - ej_aux0)
de_J02 = ej_aux2 - 2.0 * ej_aux1 + ej_aux0

# 注意：与 RHF 不同，UHF 中 `ek_aux*` 已经是 K^alpha + K^beta 的求和形式，
# 已包含全部自旋通道，因此最终公式中 K 项的系数为 -1（而非 RHF 的 -1/2），
# 这里也无需额外的 *2 转换因子。
de_K20 = ek_aux0.copy()
de_K11 = 2.0 * (ek_aux1 - ek_aux0)
de_K02 = ek_aux2 - 2.0 * ek_aux1 + ek_aux0

## CPHF 响应

UHF 的 CP-HF 方程在两个自旋通道上分别求解 `mo1a` 与 `mo1b`，两者通过 J（共享总密度响应）与 K（同自旋的交换响应）相互耦合。`mf_hess.hess_elec()` 已在内部完成这一耦合方程的求解。

In [10]:
# 5. CP-HF contribution
de_hess_elec = mf_hess.hess_elec()
de_partial = de_1 + ej_aux2 - ek_aux2
de_cphf = de_hess_elec - de_partial

## 总核验

UHF Hessian 分解公式：

$$
E_{\text{hess}} = E_{\text{nuc}} + E_{\text{1}}
    + \left(E_J^{(20)} + E_J^{(11)} + E_J^{(02)}\right)
    - \left(E_K^{(20)} + E_K^{(11)} + E_K^{(02)}\right)
    + E_{\text{cphf}}
$$

与 RHF 公式相比，K 项前的系数从 $-1/2$ 变为 $-1$（其余结构相同）。

In [11]:
de_sum = de_1 \
         + de_J20 + de_J11 + de_J02 \
         - (de_K20 + de_K11 + de_K02) \
         + de_cphf + de_nuc

print("de_ref == de_sum:", np.allclose(de_ref, de_sum))
print("max abs difference:", np.max(np.abs(de_ref - de_sum)))

de_ref == de_sum: True
max abs difference: 8.243405957841787e-15


最终，我们将这些分量都放到 `nh3_u_hf_decomp.npz` 文件中，用于后续的核验和分析。

In [12]:
dat = dict(np.load("nh3_u_hf.npz"))
dat.update({
    "de_nuc": de_nuc,
    "de_1": de_1,
    "de_J20": de_J20,
    "de_J11": de_J11,
    "de_J02": de_J02,
    "de_K20": de_K20,
    "de_K11": de_K11,
    "de_K02": de_K02,
    "de_cphf": de_cphf,
    "de_ref": de_ref,
})
np.savez("nh3_u_hf_decomp.npz", **dat)